In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline

In [3]:
# 피싱 URL 분류 파이프라인
phishing_detector = pipeline(
    "text-classification",
    model="darshan8950/phishing_url_detection_BERT"
)

sqli_detector = pipeline(
    "text-classification",
    model="cssupport/mobilebert-sql-injection-detect"
)

sms_detector = pipeline(
    "text-classification",
    model="mrm8488/bert-tiny-finetuned-sms-spam-detection"
)


config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
def security_checker(input_text):
    risk_count = 0
    risk_reasons = []

    print("=" * 70)
    print(f"[입력]: {input_text}")
    print("-" * 70)

    # 1단계: 피싱 URL 검사
    phishing_result = phishing_detector(input_text)[0]
    phishing_label = phishing_result["label"]
    phishing_score = phishing_result["score"]

    if phishing_label in ["LABEL_1", "1", "PHISHING"]:
        phishing_status = "⚠️ 피싱 위험"
        risk_count += 1
        risk_reasons.append("피싱 URL")
    else:
        phishing_status = "✅ 피싱 위험 없음"

    print(
        f"[1단계: URL 피싱 검사] → {phishing_status} "
        f"(확신도: {phishing_score:.4f})"
    )

    # 2단계: SQL 인젝션 검사
    sqli_result = sqli_detector(input_text)[0]
    sqli_label = sqli_result["label"]
    sqli_score = sqli_result["score"]

    if sqli_label == "LABEL_1":
        sqli_status = "⚠️ SQL 인젝션 감지"
        risk_count += 1
        risk_reasons.append("SQL 인젝션")
    else:
        sqli_status = "✅ SQL 인젝션 위험 없음"

    print(
        f"[2단계: SQLi 구문 검사] → {sqli_status} "
        f"(확신도: {sqli_score:.4f})"
    )

    # 3단계: 스미싱·스팸 검사
    sms_result = sms_detector(input_text)[0]
    sms_label = sms_result["label"]
    sms_score = sms_result["score"]

    if sms_label == "LABEL_1":
        sms_status = "⚠️ 스미싱·스팸 위험"
        risk_count += 1
        risk_reasons.append("스미싱·스팸")
    else:
        sms_status = "✅ 스미싱·스팸 위험 없음"

    print(
        f"[3단계: 스미싱·스팸 검사] → {sms_status} "
        f"(확신도: {sms_score:.4f})"
    )

    print("-" * 70)

    if risk_count == 0:
        print("👉 종합 판정: 탐지된 위험 요소가 없습니다.")
    else:
        reasons = ", ".join(risk_reasons)

        print(
            f"👉 종합 판정: [접근 차단] 위험 요소가 "
            f"{risk_count}건 감지되었습니다."
        )
        print(f"👉 감지 원인: {reasons}")

    print("=" * 70)
    print()

In [9]:
test_inputs = [
    # 정상 케이스
    "Hey, are we still meeting for lunch today?",

    # 피싱 URL 케이스
    "http://login-verify-account-security-alert.free-domain.com/auth/login",

    # SQL 인젝션 케이스
    "admin' OR '1'='1' --",

    # 복합 특이 케이스
    """
    URGENT! Verify your account at
    http://login-verify-account-security-alert.free-domain.com/auth/login
    ' OR '1'='1' --
    """
]

In [10]:
for text in test_inputs:
    security_checker(text)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[입력]: Hey, are we still meeting for lunch today?
----------------------------------------------------------------------
[1단계: URL 피싱 검사] → ✅ 피싱 위험 없음 (확신도: 0.8785)
[2단계: SQLi 구문 검사] → ✅ SQL 인젝션 위험 없음 (확신도: 0.9975)
[3단계: 스미싱·스팸 검사] → ✅ 스미싱·스팸 위험 없음 (확신도: 0.9368)
----------------------------------------------------------------------
👉 종합 판정: 탐지된 위험 요소가 없습니다.

[입력]: http://login-verify-account-security-alert.free-domain.com/auth/login
----------------------------------------------------------------------
[1단계: URL 피싱 검사] → ✅ 피싱 위험 없음 (확신도: 1.0000)
[2단계: SQLi 구문 검사] → ✅ SQL 인젝션 위험 없음 (확신도: 0.9958)
[3단계: 스미싱·스팸 검사] → ⚠️ 스미싱·스팸 위험 (확신도: 0.6571)
----------------------------------------------------------------------
👉 종합 판정: [접근 차단] 위험 요소가 1건 감지되었습니다.
👉 감지 원인: 스미싱·스팸

[입력]: admin' OR '1'='1' --
----------------------------------------------------------------------
[1단계: URL 피싱 검사] → ✅ 피싱 위험 없음 (확신도: 0.9998)
[2단계: SQLi 구문 검사] → ⚠️ SQL 인젝션 감지 (확신도: 1.0000)
[3단계: 스미싱·스팸 검사] → ✅ 스미싱·스팸 위험 없음 (확신도: